In [7]:
import torch
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [8]:
max_seq_len = 4096
adapter_path = "adapters/echobot_pilot"

lora_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = adapter_path,
    max_seq_length = max_seq_len,
    dtype = None,
    load_in_4bit = True 
)

==((====))==  Unsloth 2026.8.12: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [9]:
FastLanguageModel.for_inference(lora_model)

system_prompt = "You are a classifier for another chatbot. Your job is to determine if the user's query is appropriate for the task of archival search. If the user query is appropriate for archival search, respond with a single character '0'. If the user query is not appropriate, respond with a single character '1'."
test_query = "Find me Echo articles that discuss past registrars and major changes to the grading scale"

prompt = f"System: {system_prompt}\nUser: {test_query}\nAssistant: "

inputs = tokenizer(
    text=[prompt], 
    return_tensors="pt"
).to("cuda")

print(f"{prompt=}\n")
print(f"{inputs=}\n")

prompt="System: You are a classifier for another chatbot. Your job is to determine if the user's query is appropriate for the task of archival search. If the user query is appropriate for archival search, respond with a single character '0'. If the user query is not appropriate, respond with a single character '1'.\nUser: Find me Echo articles that discuss past registrars and major changes to the grading scale\nAssistant: "

inputs={'input_ids': tensor([[ 2244,    25,  1394,   513,   264, 32260,   364,  2361,  6040,  6132,
            13,  4465,  2531,   369,   310,  7995,   413,   279,  1156,   579,
          3134,   369,  8053,   364,   279,  3274,   314, 90702,  2624,    13,
          1368,   279,  1156,  3134,   369,  8053,   364, 90702,  2624,    11,
          5707,   440,   264,  3074,  3542,   359,    15,  4282,  1368,   279,
          1156,  3134,   369,   524,  8053,    11,  5707,   440,   264,  3074,
          3542,   359,    16,  4282,   198,  1421,    25,  7145,   728, 3655

In [ ]:
outputs = lora_model.generate(
    **inputs,
    max_new_tokens=2,
    use_cache=True,
    pad_token_id=tokenizer.eos_token_id
)
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print(response[-3:].strip())

# Batch Tests


In [ ]:
max_seq_len = 4096
adapter_path = "adapters/echobot_pilot"

lora_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = adapter_path,
    max_seq_length = max_seq_len,
    dtype = None,
    load_in_4bit = True 
)

FastLanguageModel.for_inference(lora_model)

In [ ]:
import json

test_queries = []
test_labels = []

with open("./data/echobot_test.jsonl", "r") as json_file:
    jsonl_data = list(json_file)

for json_str in jsonl_data:
    sample = json.loads(json_str)
    test_queries.append(sample['query'])
    test_labels.append(sample['label'])

print(len(test_queries))
print(len(test_labels))
print()
print(test_queries[0])
print(test_labels[0])

In [15]:
system_prompt = "You are a classifier for another chatbot. Your job is to determine if the user's query is appropriate for the task of archival search. If the user query is appropriate for archival search, respond with a single character '0'. If the user query is not appropriate, respond with a single character '1'."

num_correct = 0
num_wrong = 0
false_positives = 0
false_negatives = 0

for label, test_query in zip(test_labels, test_queries):
    prompt = f"System: {system_prompt}\nUser: {test_query}\nAssistant: "

    inputs = tokenizer(
        text=[prompt], 
        return_tensors="pt"
    ).to("cuda")

    outputs = lora_model.generate(
        **inputs,
        max_new_tokens=2,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    response = response[-3:].strip()
    if str(response) == str(label):
        num_correct += 1
    else: 
        num_wrong += 1
        if label == "1":
            false_negatives += 1
        else:
            false_positives += 1
    
print(f"{(num_correct / (num_correct + num_wrong)) * 100:.2f}% Accuracy on test set")
print(f"{false_positives} false positives | {false_negatives} false negatives")

50.00% Accuracy on test set
8 false positives | 0 false negatives
